In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244081109
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] py311
[学号] 20244081109
[机器指纹] 21935ea21d1fabf1
[RUN_ID] 20244081109-21935ea21d1fabf1
[Python] 3.11.13


# 二、实验内容与过程

## 实验 1：正则表达式的基础匹配

先导入 `re`。三个任务分别演示**字符类**、**选择分支**和**量词**的作用：`\d` 数字、`\s` 空白、`\b` 单词边界；`|` 按从左到右的顺序匹配；`*` `+` `?` `{m}` `{m,n}` 控制前一个元素重复的次数。

任务 1：在 "cat on mat; later: cats and dogs" 上分别匹配 `\d+`、`\s+`、`\bcat\b`。

In [2]:
import re

text = "cat on mat; later: cats and dogs"

print("1) \\d+  数字      ->", re.findall(r"\d+", text))
print("2) \\s+  空白      ->", re.findall(r"\s+", text))
print("   用 repr 看清内容 ->", [repr(s) for s in re.findall(r"\s+", text)])
print("3) \\bcat\\b 独立单词 ->", re.findall(r"\bcat\b", text))
print("   对照：不加边界的 cat ->", re.findall(r"cat", text))

1) \d+  数字      -> []
2) \s+  空白      -> [' ', ' ', ' ', ' ', ' ', ' ']
   用 repr 看清内容 -> ["' '", "' '", "' '", "' '", "' '", "' '"]
3) \bcat\b 独立单词 -> ['cat']
   对照：不加边界的 cat -> ['cat', 'cat']


结果解读：
1. `\d+` 返回空列表：这段文本里根本没有数字，正则"匹配不到不报错，只返回空结果"。
2. `\s+` 返回 6 个空格。直接打印时列表里是一串看不见的空白，所以用 `repr` 才看得出每一项是 `' '`（一个空格）；`\s` 匹配空格、制表符、换行等所有空白字符，`+` 表示连续一段算一个匹配。
3. `\bcat\b` 只返回 `['cat']`：`\b` 是单词边界（单词字符与非单词字符之间的位置），所以 cats 里的 cat 前面虽然能对上，但后面紧跟 `s`（仍是单词字符），不是边界，因此不匹配。对照不加边界的 `cat` 得到 `['cat', 'cat']`，第二个正是 "cats" 里的那三个字母——这就是 `\b` 的意义。

任务 2：在 "cat dog bird CAT" 上用 `cat|dog`；再用 `re.findall(r"a|aa|aaa", "aaaa")` 和 `re.findall(r"aaa|aa|a", "aaaa")` 展示差异。

In [3]:
text2 = "cat dog bird CAT"

print("cat|dog          ->", re.findall(r"cat|dog", text2))
print("cat|dog|CAT      ->", re.findall(r"cat|dog|CAT", text2))
print("cat|dog + IGNORECASE ->", re.findall(r"cat|dog", text2, flags=re.IGNORECASE))

print()
print("a|aa|aaa  on 'aaaa' ->", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a  on 'aaaa' ->", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog          -> ['cat', 'dog']
cat|dog|CAT      -> ['cat', 'dog', 'CAT']
cat|dog + IGNORECASE -> ['cat', 'dog', 'CAT']

a|aa|aaa  on 'aaaa' -> ['a', 'a', 'a', 'a']
aaa|aa|a  on 'aaaa' -> ['aaa', 'a']


结果解读：
1. `cat|dog` 只匹配到 `cat` 和 `dog`，`CAT` 没匹配：正则默认区分大小写；加上 `re.IGNORECASE` 后 `CAT` 也能命中，返回的是原文本里的写法 `CAT`。
2. 两个分支顺序不同，结果完全不同：`a|aa|aaa` 在 `"aaaa"` 上得到 4 个 `'a'`，`aaa|aa|a` 得到 `'aaa'` 和 `'a'`。原因是 Python 的 `|` 是"最左优先，同一位置按分支书写顺序取第一个能匹配的"：扫描到第 0 个字符时，`a|aa|aaa` 里 `a` 先试成功就收工，于是每次只吃一个 a；而 `aaa|aa|a` 里 `aaa` 先试成功，一次吃掉三个。这提醒我们：写分支时长的、具体的要放在前面。

任务 3：在 "ct cat caat caaat" 上分别使用 `ca*t`、`ca+t`、`ca?t`、`ca{2}t` 和 `ca{1,2}t`，输出匹配列表并说明各量词的含义。


In [4]:
text3 = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

for p in patterns:
    print(f"{p:<10} -> {re.findall(p, text3)}")

ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


结果解读（量词含义）：
| 量词 | 含义 | 在 "ct cat caat caaat" 上 |
|---|---|---|
| `*` | 前面的 a 重复 0 次或多次 | 四种全中：`ct` `cat` `caat` `caaat` |
| `+` | 重复 1 次或多次 | `cat` `caat` `caaat`（`ct` 里没有 a，不匹配） |
| `?` | 重复 0 次或 1 次 | 只有 `ct`、`cat`（多于一个 a 就对不上 `t`） |
| `{2}` | 恰好 2 次 | 只有 `caat` |
| `{1,2}` | 1 到 2 次（能多则多，贪婪） | `cat`、`caat`；`caaat` 有 3 个 a，超出上限，不匹配 |

可以看出量词只管紧挨着它左边的那一个元素（这里是 `a`），而 `c` 和 `t` 是必须出现的字面字符。


## 实验 2：分组、后向引用与替换

`()` 是捕获组，可以用 `group(n)` 取出来，也可以在替换串里用 `\1` `\2` 引用；`\1` 还能在**模式内部**回头引用前面捕获到的内容，用来找重复。

任务 1：在 "联系：alice@example.com；备选：bob@test.org" 中用捕获组提取每个邮箱地址的用户名和域名。


In [5]:
text = "联系：alice@example.com；备选：bob@test.org"

pairs = re.findall(r"([\w.]+)@([\w.-]+)", text)
print("findall 直接得到元组列表：", pairs)

for m in re.finditer(r"([\w.]+)@([\w.-]+)", text):
    print(f"整体={m.group(0):<20} 用户名={m.group(1):<6} 域名={m.group(2):<12} 位置={m.span()}")

findall 直接得到元组列表： [('alice', 'example.com'), ('bob', 'test.org')]
整体=alice@example.com    用户名=alice  域名=example.com  位置=(3, 20)
整体=bob@test.org         用户名=bob    域名=test.org     位置=(24, 36)


结果解读：`findall` 遇到有捕获组的模式时，返回的是元组列表，而不是整段匹配的字符串，所以直接得到 `[('alice', 'example.com'), ('bob', 'test.org')]`；想同时看到整体匹配和位置就用 `finditer`。用户名用 `[\w.]+`（允许字母数字下划线和点，如 `first.last`），域名用 `[\w.-]+`（多加一个 `-`，因为域名可以带连字符）。

任务 2：在 "cat cat, dog cat, bird bird" 中用后向引用找出连续重复的单词。


In [6]:
text = "cat cat, dog cat, bird bird"

print("重复单词：", re.findall(r"\b(\w+)\s+\1\b", text))

for m in re.finditer(r"\b(\w+)\s+\1\b", text):
    print(f"命中 {m.group(0)!r:<12} 重复的词 = {m.group(1)!r}")

重复单词： ['cat', 'bird']
命中 'cat cat'    重复的词 = 'cat'
命中 'bird bird'  重复的词 = 'bird'


结果解读：模式 `\b(\w+)\s+\1\b` 的意思是"单词边界 + 捕获一个单词 + 若干空白 + 和刚捕获完全相同的那个词（`\1` 后向引用）+ 单词边界"。所以 `cat cat` 和 `bird bird` 命中，`dog cat`（两个不同词）不会命中——`\1` 保证了内容相同，而不是"任意两个词"。用 `\w+` 而不是 `\w`，才能比较整个单词。

任务 3：将 "2024-05-01\n2024/05/02\n2024.05.03" 中的日期从"年-月-日"转换为"月-日-年"，保留原来的行数。
解答思路：用 `re.sub(pattern, repl, text)` 一次替换全部匹配；三个捕获组分别是年、月、日，替换串写成 `\2-\3-\1` 即可换位；`[-/.]` 是字符类，`-` 放在最前面就不用转义，三种分隔符一并覆盖。`sub` 只改匹配到的片段，换行符不动，所以行数不变。

In [7]:
text = "2024-05-01\n2024/05/02\n2024.05.03"

print("转换前：")
print(text)

new_text = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", text)

print("\n转换后：")
print(new_text)

print("\n行数：", len(text.splitlines()), "->", len(new_text.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024

行数： 3 -> 3


**任务 4：**将 "联系我：123-456-7890 或 987.654.3210" 中的电话号码统一为 (123) 456-7890 形式。

In [8]:
text = "联系我：123-456-7890 或 987.654.3210"

new_text = re.sub(r"(\d{3})[.-](\d{3})[.-](\d{4})", r"(\1) \2-\3", text)
print(new_text)

# 再验证一次：替换后的号码还能被同一个模式识别
print("是否已被格式化：", re.findall(r"\(\d{3}\) \d{3}-\d{4}", new_text))

联系我：(123) 456-7890 或 (987) 654-3210
是否已被格式化： ['(123) 456-7890', '(987) 654-3210']


**结果解读：**三段数字分别捕获成区号、前三位、后四位，替换串 `(\1) \2-\3` 里 `\1` `\2` `\3` 是**反向引用**，括号和空格则是普通字符，所以两组号码（一个用 `-`、一个用 `.`）都变成了统一的 `(123) 456-7890` 样式。注意字符类里的 `-` 如果不放在首尾，会被当成范围（如 `[0-9]`），这里写在中间必须放末尾或转义。

## 实验 3：使用 BeautifulSoup 解析 HTML

题目给出固定的 HTML 字符串，用 `BeautifulSoup(html_doc, "html.parser")` 建树即可，不联网下载。`html.parser` 是 Python 标准库解析器，不用额外装 lxml。

任务 1：输出页面标题文本、h1 的标签名与文本。


In [9]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题：", soup.title.get_text())
h1 = soup.h1
print("h1 标签名：", h1.name)
print("h1 文本：", h1.get_text())

页面标题： 实验课程
h1 标签名： h1
h1 文本： 数据分析


结果解读：`soup.title` 直接取到 `<title>` 节点（`soup.title` 等价于 `soup.find("title")`），`.get_text()` 只取标签内的文字、去掉标签本身；`h1.name` 返回标签名字符串 `'h1'`，`.get_text()` 返回 `'数据分析'`，可见标签名和文本是两个不同的属性。


任务 2：用 `find_all()` 找出所有 `class="course"` 的段落和所有链接，逐项输出链接文字与 href 属性。


In [10]:
paras = soup.find_all("p", class_="course")
print(f"class=course 的段落：{len(paras)} 个")
for i, p in enumerate(paras, 1):
    print(f"  {i}. 标签={p.name}  class={p['class']}  文本={p.get_text()}")

links = soup.find_all("a")
print(f"\n链接：{len(links)} 个")
for i, a in enumerate(links, 1):
    print(f"  {i}. 文字={a.get_text()}  href={a['href']}")

print("\n只保留有 href 的链接：", [(a.get_text(), a["href"]) for a in soup.find_all("a", href=True)])

class=course 的段落：2 个
  1. 标签=p  class=['course']  文本=第六讲：正则表达式
  2. 标签=p  class=['course']  文本=第七讲：结构化数据

链接：2 个
  1. 文字=讲义六  href=https://example.org/lecture06
  2. 文字=讲义七  href=https://example.org/lecture07

只保留有 href 的链接： [('讲义六', 'https://example.org/lecture06'), ('讲义七', 'https://example.org/lecture07')]


结果解读：`find_all("p", class_="course")` 返回的是 ResultSet（可以当列表用），筛出两个段落；`class` 是 Python 关键字，所以 BeautifulSoup 用 `class_` 这个参数名，值可以是字符串、列表或正则。取属性用 `tag["href"]`，取不到会抛 `KeyError`，稳妥写法是 `tag.get("href")`。

任务 3：用 `get_text()` 提取页面可见文本，比较整个文档、单个段落和单个链接的文本结果。


In [11]:
print("整个文档：", repr(soup.get_text()))
print("单个段落：", repr(paras[0].get_text()))
print("单个链接：", repr(links[0].get_text()))

print("\n规整后的整页文本：")
print(soup.get_text(separator="|", strip=True))

print("\n可见文本（排除 head 里的 title）：")
print(soup.body.get_text(separator="|", strip=True))

整个文档： '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
单个段落： '第六讲：正则表达式'
单个链接： '讲义六'

规整后的整页文本：
实验课程|数据分析|第六讲：正则表达式|第七讲：结构化数据|讲义六|讲义七

可见文本（排除 head 里的 title）：
数据分析|第六讲：正则表达式|第七讲：结构化数据|讲义六|讲义七


结果解读：
1. `soup.get_text()` 会把整棵树的文本节点拼起来，标签之间的换行、缩进会原样保留，所以 `repr` 里能看到大量 `\n` 和空白，还包含 `<head>` 里的 title。
2. `paras[0].get_text()` 只包含这一个 `<p>` 的内容 `'第六讲：正则表达式'`，干净的单行。
3. `links[0].get_text()` 只返回 `<a>` 里的可见文字 `'讲义六'`，不含 `href` 属性——属性值不是文本节点。
4. 想要规整结果可以加参数：`separator="|"` 用指定分隔符连接，`strip=True` 去掉首尾空白；只想看正文就用 `soup.body.get_text(...)`，这样不会把 title 混进来。
